# LeRobot 本地数据集 Feature 重命名

> 环境：**myvla**（官方 `/vla/my_vla/src/lerobot`）

本 notebook 实现本地 v3 数据集的 feature key 重命名，会同步更新：
- `meta/info.json` features
- `data/**/*.parquet` 列名（若有）
- `videos/<key>/` 目录名（video 数据集）
- `meta/episodes/**/*.parquet` 中的 `videos/<key>/...` 列名（**关键**）
- `meta/stats.json` 统计 key

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

import shutil
from pathlib import Path
from typing import Dict

import pandas as pd
from tqdm import tqdm

from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.utils import write_info, write_stats, write_tasks

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _rename_episode_metadata_columns(columns: list[str], rename_map: Dict[str, str]) -> Dict[str, str]:
    """重命名 meta/episodes parquet 里 videos/<key>/... 或 images/<key>/... 列名。"""
    col_rename = {}
    for col in columns:
        new_col = col
        for old_key, new_key in rename_map.items():
            for media_prefix in ("videos/", "images/"):
                old_prefix = f"{media_prefix}{old_key}/"
                if col.startswith(old_prefix):
                    new_col = f"{media_prefix}{new_key}/" + col[len(old_prefix):]
                    break
            if new_col != col:
                break
        if new_col != col:
            col_rename[col] = new_col
    return col_rename


def rename_features_in_dataset(
    original_path: str | Path,
    new_path: str | Path,
    rename_map: Dict[str, str],
    overwrite: bool = False,
) -> LeRobotDataset:
    """本地 LeRobot v3 数据集 feature 重命名，输出到新目录。"""
    original_path = Path(original_path).resolve()
    new_path = Path(new_path).resolve()

    if not original_path.is_dir():
        raise FileNotFoundError(f"源数据集不存在: {original_path}")
    if new_path.exists():
        if not overwrite:
            raise FileExistsError(f"目标目录已存在: {new_path}，设 overwrite=True 或先删除")
        shutil.rmtree(new_path)

    ds = LeRobotDataset(repo_id=str(original_path), download_videos=False)
    print(f"加载: {original_path}  ({ds.num_frames} frames)")

    for old_key in rename_map:
        if old_key not in ds.meta.features:
            raise KeyError(f"rename_map 中的 key 不存在: {old_key}")

    new_features = {}
    for old_key, feat_info in ds.meta.features.items():
        new_key = rename_map.get(old_key, old_key)
        if new_key in new_features:
            raise ValueError(f"重命名后 key 冲突: {new_key}")
        new_features[new_key] = feat_info

    print("重命名映射:")
    for old_key, new_key in rename_map.items():
        print(f"  {old_key}  →  {new_key}")

    new_meta = LeRobotDatasetMetadata.create(
        repo_id=new_path.name,
        fps=ds.meta.fps,
        features={k: v for k, v in new_features.items()
                  if k not in {"timestamp", "frame_index", "episode_index", "index", "task_index"}},
        robot_type=ds.meta.robot_type,
        root=new_path,
        use_videos=len(ds.meta.video_keys) > 0,
    )

    parquet_files = sorted((original_path / "data").glob("*/*.parquet"))
    col_rename = {old: new for old, new in rename_map.items()}
    for src_path in tqdm(parquet_files, desc="data parquet"):
        rel = src_path.relative_to(original_path / "data")
        dst_path = new_path / "data" / rel
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        df = pd.read_parquet(src_path)
        df = df.rename(columns={k: v for k, v in col_rename.items() if k in df.columns})
        df.to_parquet(dst_path, index=False)

    src_videos = original_path / "videos"
    if src_videos.exists():
        for video_key in ds.meta.video_keys:
            new_key = rename_map.get(video_key, video_key)
            src_dir = src_videos / video_key
            if src_dir.exists():
                shutil.copytree(src_dir, new_path / "videos" / new_key)

    src_images = original_path / "images"
    if src_images.exists():
        for img_key in ds.meta.image_keys:
            new_key = rename_map.get(img_key, img_key)
            src_dir = src_images / img_key
            if src_dir.exists():
                shutil.copytree(src_dir, new_path / "images" / new_key)

    # episodes 元数据：必须同步重命名 videos/<key>/... 列
    src_episodes = original_path / "meta" / "episodes"
    if src_episodes.exists():
        ep_files = sorted(src_episodes.rglob("*.parquet"))
        for src_path in tqdm(ep_files, desc="episodes parquet"):
            rel = src_path.relative_to(src_episodes)
            dst_path = new_path / "meta" / "episodes" / rel
            dst_path.parent.mkdir(parents=True, exist_ok=True)
            df = pd.read_parquet(src_path)
            ep_col_rename = _rename_episode_metadata_columns(list(df.columns), rename_map)
            df = df.rename(columns=ep_col_rename)
            df.to_parquet(dst_path, index=False)

    if ds.meta.tasks is not None:
        write_tasks(ds.meta.tasks, new_path)

    new_info = dict(ds.meta.info)
    new_info["features"] = new_features
    write_info(new_info, new_path)

    if ds.meta.stats:
        new_stats = {rename_map.get(k, k): v for k, v in ds.meta.stats.items()}
        write_stats(new_stats, new_path)

    new_ds = LeRobotDataset(repo_id=str(new_path), download_videos=False)
    print(f"\n✅ 完成: {new_path}")
    print("新 camera keys:", new_ds.meta.camera_keys)
    return new_ds

In [3]:
# ====================== 配置区 ======================
ORIGINAL_PATH = Path("/vla/.data/test")
NEW_PATH = Path("/vla/.data/test_rename")

RENAME_MAP = {
    "observation.images.robot0_agentview_left_rgb": "observation.images.image",
    "observation.images.robot0_agentview_right_rgb": "observation.images.image2",
    "observation.images.robot0_eye_in_hand_rgb": "observation.images.image3",
}

new_ds = rename_features_in_dataset(
    original_path=ORIGINAL_PATH,
    new_path=NEW_PATH,
    rename_map=RENAME_MAP,
    overwrite=True,  # 上次失败可能留下半成品目录
)

加载: /vla/.data/test  (22733 frames)
重命名映射:
  observation.images.robot0_agentview_left_rgb  →  observation.images.image
  observation.images.robot0_agentview_right_rgb  →  observation.images.image2
  observation.images.robot0_eye_in_hand_rgb  →  observation.images.image3


episodes parquet: 100%|██████████| 1/1 [00:00<00:00, 16.40it/s]



✅ 完成: /vla/.data/test_rename
新 camera keys: ['observation.images.image', 'observation.images.image2', 'observation.images.image3']


In [4]:
# 验证
sample = new_ds[0]
print("camera keys:", new_ds.meta.camera_keys)
print("sample keys:", sorted(sample.keys()))
for k in new_ds.meta.camera_keys:
    print(f"  {k}: {sample[k].shape}")

camera keys: ['observation.images.image', 'observation.images.image2', 'observation.images.image3']
sample keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.images.image', 'observation.images.image2', 'observation.images.image3', 'observation.state', 'task', 'task_index', 'timestamp']
  observation.images.image: torch.Size([3, 224, 224])
  observation.images.image2: torch.Size([3, 224, 224])
  observation.images.image3: torch.Size([3, 224, 224])
